### Load the Model

In [1]:
from gpt_model import GPTModel
gpt_config = {
    "vocab_size": 50257,
    "context_length": 256, # new context length
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

model = GPTModel(gpt_config)

/Users/rishidinesh/Projects/llm-from-scratch/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


tensor([[[-0.1788, -0.6077],
         [-0.2112, -0.6468],
         [-0.2210, -0.6612],
         [-0.1933, -0.6523],
         [-0.1757, -0.6634],
         [-0.1714, -0.6525]],

        [[-0.1788, -0.6077],
         [-0.2112, -0.6468],
         [-0.2210, -0.6612],
         [-0.1933, -0.6523],
         [-0.1757, -0.6634],
         [-0.1714, -0.6525]]], grad_fn=<ViewBackward0>)
torch.Size([2, 6, 2])
tensor([[15496,    11,   314,   716]]) torch.Size([1, 4])
Hello, I am scanningStudio Chocobo systematic Sanskrit Schools chops chemotherapy sabotage veil


### Define utility functions

In [2]:
import torch
import tiktoken
from gpt_model import generate_text

torch.manual_seed(42)

tokenizer = tiktoken.get_encoding("gpt2")

def text_to_token_ids(text):
    tokens = tokenizer.encode(
        text = text,
        allowed_special = {'<|endoftext|>'}
        )
    return torch.tensor(tokens).unsqueeze(0)

def token_ids_to_text(token_ids):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

### Make predictions on sample input/target pairs

In [3]:
inputs = torch.tensor([
    [16833, 3626, 6100], # every effor moves
    [40, 1107, 588] # I really like
])

targets = torch.tensor([
    [3626, 6100, 345], # effort moves you
    [1107, 588, 11311] # really like chocolate
])

In [4]:
model = GPTModel(gpt_config)
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim = -1)
token_ids = torch.argmax(probas, dim = -1)

In [5]:
for i, t_ids in enumerate(token_ids):
    print(f"Inputs: {token_ids_to_text(inputs[i])}")
    text = token_ids_to_text(t_ids)
    print(f"Predictions: {text}")
    print(f"Targets: {token_ids_to_text(targets[i])}")

Inputs: every effort moves
Predictions:  Vi Heller HO
Targets:  effort moves you
Inputs: I really like
Predictions: ovsky dummyogenesis
Targets:  really like chocolate


### Calculate sample loss

In [6]:
target_prob_1 = probas[0, [0, 1, 2], targets[0]]
target_prob_2 = probas[0, [0, 1, 2], targets[1]]
logprobs = torch.log(torch.cat((target_prob_1, target_prob_2)))
print(logprobs)
-1 * torch.mean(logprobs)

tensor([-12.5472, -10.7895, -11.8781, -11.3533, -10.7056, -11.9942])


tensor(11.5447)

The above is the same as doing:

In [7]:
from torch.nn import functional as F

logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print(logits_flat.shape, targets_flat.shape)
F.cross_entropy(logits_flat, targets_flat)

torch.Size([6, 50257]) torch.Size([6])


tensor(11.3987)

## Setting up training and validation losses

### Load Data

In [8]:
import os
import urllib.request

if not os.path.exists("./the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
    file_path = "./the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

with open("./the-verdict.txt", "r", encoding="utf-8") as f:
    text = f.read()

### Set up data loaders

In [9]:
split = 0.9
split_idx = int(0.9 * len(text))
train = text[:split_idx]
val = text[split_idx:]

In [10]:
from dataloader import create_dataloader

train_loader = create_dataloader(
    text = train,
    batch_size = 2,
    max_length = gpt_config["context_length"],
    stride = gpt_config["context_length"],
    drop_last = True,
    shuffle = True
)

val_loader = create_dataloader(
    text = val,
    batch_size = 2,
    max_length = gpt_config["context_length"],
    stride = gpt_config["context_length"],
    drop_last = False,
    shuffle = False
)

In [11]:
for x, y in train_loader:
    print(x.shape, y.shape)
    print(x[0][:10], y[0][:10], sep = "\n")
    break

torch.Size([2, 256]) torch.Size([2, 256])
tensor([  198,  1544, 13818,  4622,    11,  1231, 35987,    11,   290,  7121])
tensor([ 1544, 13818,  4622,    11,  1231, 35987,    11,   290,  7121,   530])


### Calculate average loss

In [12]:
from torch.nn import functional as F

# calculated over a batch
def calculate_batch_loss(input, target, model, device):
    input = input.to(device)
    target = target.to(device)
    logits = model(input)
    loss = F.cross_entropy(logits.flatten(0, 1), target.flatten())
    return loss

In [13]:
# calculated over a given loader for num_batches
def get_average_loss(data_loader, model, device, num_batches = None):
    total_loss = 0
    if len(data_loader) == 0:
        return float("nan")
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (xb, yb) in enumerate(data_loader):
        if i < num_batches:
            batch_loss = calculate_batch_loss(xb, yb, model, device)
            total_loss += batch_loss.item()
        else:
            break
    return total_loss / num_batches

In [14]:
device = "mps"
model.to(device)
with torch.no_grad():
    train_loss = get_average_loss(train_loader, model, device)
    val_loss = get_average_loss(val_loader, model, device)
    print(f"Training loss: {train_loss}\nValidation loss: {val_loss}")

Training loss: 11.004175927903917
Validation loss: 11.030633926391602


## Setup training loop

In [15]:
def generate_and_print_sample(model, device, start_context):
    model.eval()
    context_size = model.config["context_length"]
    encoded = text_to_token_ids(start_context).to(device)
    with torch.no_grad():
        token_ids = generate_text(model, encoded, max_new_tokens=50, context_size = context_size)
    res = token_ids_to_text(token_ids)
    print(res.replace("\n", ""))
    model.train()

In [16]:
def train_model(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context):
    train_losses, val_losses = [], []
    num_tokens_seen, global_step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = calculate_batch_loss(xb, yb, model, device)
            loss.backward()
            optimizer.step()
            num_tokens_seen += xb.numel()
            global_step += 1
            if global_step % eval_freq == 0:
                model.eval()
                with torch.no_grad():
                    train_loss = get_average_loss(train_loader, model, device, eval_iter)
                    val_loss = get_average_loss(val_loader, model, device, eval_iter)
                    train_losses.append(train_loss)
                    val_losses.append(val_loss)
                model.train()
        print(f"Epoch: {epoch+1} | Step: {global_step} | Tokens seen : {num_tokens_seen} | train loss: {train_loss:.3f} | val loss: {val_loss:.3f}")
        generate_and_print_sample(model, device, start_context)
    return train_losses, val_losses

In [17]:
optimizer = torch.optim.AdamW(params = model.parameters(), lr = 0.0004, weight_decay = 0.1)
num_epochs = 10
train_losses, val_losses = train_model(
    model, train_loader, val_loader, optimizer, device, num_epochs, 5, 5, "Every effort moves you"
)

Epoch: 1 | Step: 8 | Tokens seen : 4608 | train loss: 8.062 | val loss: 8.336
Every effort moves you.
Epoch: 2 | Step: 17 | Tokens seen : 9216 | train loss: 5.990 | val loss: 6.645
Every effort moves you, the.
Epoch: 3 | Step: 26 | Tokens seen : 13824 | train loss: 5.338 | val loss: 6.425
Every effort moves you of the of the a little, and of the, and
Epoch: 4 | Step: 35 | Tokens seen : 18432 | train loss: 4.647 | val loss: 6.393
Every effort moves you the"I. Gisburn was the picture--I the the"I was the picture the picture--as, and he had been the picture, and"I, and he was the picture was the picture--as the picture was the"
Epoch: 5 | Step: 44 | Tokens seen : 23040 | train loss: 4.365 | val loss: 6.312
Every effort moves you the picture was not that he was dead.""--as--as the donkey, and."--and--and--as the picture
Epoch: 6 | Step: 53 | Tokens seen : 27648 | train loss: 3.289 | val loss: 6.309
Every effort moves you know you know."--I the me--and."--and I had the donkey."--and the his